In [1]:
# @title Part-3.1_QSAR Descriptor Generator

"""
Advanced QSAR Descriptor Generator for Google Colab
===================================================

This script generates comprehensive molecular descriptors for QSAR modeling.
It includes multiple descriptor types with advanced features and error handling.

Author: Advanced QSAR Implementation
"""

# Install required packages
!pip install rdkit-pypi mordred pandas numpy scikit-learn networkx tqdm

# Import necessary libraries
import pandas as pd
import numpy as np
import networkx as nx
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, Crippen, Lipinski
from rdkit.Chem import rdmolops, QED, Fragments
from rdkit.Chem.EState import Fingerprinter
from rdkit.Chem.Fingerprints import FingerprintMols
from rdkit.Chem.AtomPairs import Pairs
from rdkit.Chem.AtomPairs import Torsions
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

class AdvancedDescriptorGenerator:
    """
    Advanced molecular descriptor generator for QSAR modeling
    """

    def __init__(self):
        self.descriptor_names = []
        self.failed_molecules = []

    def sanitize_molecule(self, mol):
        """Sanitize molecule and handle errors"""
        try:
            Chem.SanitizeMol(mol)
            return mol
        except:
            return None

    def generate_3d_conformer(self, mol):
        """Generate 3D conformer with improved algorithm"""
        try:
            mol = Chem.AddHs(mol)
            # Use ETKDG algorithm for better conformer generation
            AllChem.EmbedMolecule(mol,
                                randomSeed=42,
                                useExpTorsionAnglePrefs=True,
                                useBasicKnowledge=True)
            # Use MMFF94 force field for optimization
            AllChem.MMFFOptimizeMolecule(mol, maxIters=1000)
            return mol
        except:
            return None

    def calculate_topological_descriptors(self, mol):
        """Calculate advanced topological descriptors"""
        descriptors = {}

        try:
            # Basic topological descriptors
            descriptors['WienerIndex'] = self._wiener_index(mol)
            descriptors['ZagrebIndex1'] = self._zagreb_index_1(mol)
            descriptors['ZagrebIndex2'] = self._zagreb_index_2(mol)
            descriptors['BalabanJ'] = Descriptors.BalabanJ(mol)
            descriptors['TPSA'] = Descriptors.TPSA(mol)

            # Kappa shape indices
            descriptors['Kappa1'] = Descriptors.Kappa1(mol)
            descriptors['Kappa2'] = Descriptors.Kappa2(mol)
            descriptors['Kappa3'] = Descriptors.Kappa3(mol)

            # Chi connectivity indices
            descriptors['Chi0'] = Descriptors.Chi0(mol)
            descriptors['Chi1'] = Descriptors.Chi1(mol)
            descriptors['Chi0n'] = Descriptors.Chi0n(mol)
            descriptors['Chi1n'] = Descriptors.Chi1n(mol)
            descriptors['Chi2n'] = Descriptors.Chi2n(mol)
            descriptors['Chi3n'] = Descriptors.Chi3n(mol)
            descriptors['Chi4n'] = Descriptors.Chi4n(mol)

            # Hall-Kier alpha
            descriptors['HallKierAlpha'] = Descriptors.HallKierAlpha(mol)

            # Custom Petitjean number calculation (since it's not available in older RDKit)
            descriptors['PetitjeanNumber'] = self._petitjean_number(mol)

        except Exception as e:
            print(f"Error calculating topological descriptors: {e}")

        return descriptors

    def _wiener_index(self, mol):
        """Calculate Wiener index using networkx"""
        try:
            adj_matrix = rdmolops.GetAdjacencyMatrix(mol)
            G = nx.from_numpy_array(adj_matrix)
            return nx.wiener_index(G)
        except:
            return np.nan

    def _zagreb_index_1(self, mol):
        """Calculate first Zagreb index"""
        try:
            adj_matrix = rdmolops.GetAdjacencyMatrix(mol)
            degrees = adj_matrix.sum(axis=0)
            return np.sum(degrees**2)
        except:
            return np.nan

    def _zagreb_index_2(self, mol):
        """Calculate second Zagreb index"""
        try:
            adj_matrix = rdmolops.GetAdjacencyMatrix(mol)
            degrees = adj_matrix.sum(axis=0)
            zagreb2 = 0
            for i in range(len(degrees)):
                for j in range(i+1, len(degrees)):
                    if adj_matrix[i,j] == 1:
                        zagreb2 += degrees[i] * degrees[j]
            return zagreb2
        except:
            return np.nan

    def _petitjean_number(self, mol):
        """Calculate Petitjean number manually"""
        try:
            adj_matrix = rdmolops.GetAdjacencyMatrix(mol)
            G = nx.from_numpy_array(adj_matrix)
            if G.number_of_nodes() <= 2:
                return 0

            # Calculate all shortest path lengths
            all_shortest_paths = dict(nx.all_pairs_shortest_path_length(G))

            # Find diameter (maximum distance)
            diameter = 0
            radius = float('inf')

            for source in all_shortest_paths:
                max_dist = max(all_shortest_paths[source].values())
                diameter = max(diameter, max_dist)
                radius = min(radius, max_dist)

            if diameter == 0:
                return 0

            return (diameter - radius) / diameter
        except:
            return np.nan

    def _calculate_fraction_csp3(self, mol):
        """Calculate fraction of sp3 carbons manually"""
        try:
            carbon_count = 0
            sp3_carbon_count = 0

            for atom in mol.GetAtoms():
                if atom.GetAtomicNum() == 6:  # Carbon
                    carbon_count += 1
                    if atom.GetHybridization() == Chem.HybridizationType.SP3:
                        sp3_carbon_count += 1

            if carbon_count == 0:
                return 0

            return sp3_carbon_count / carbon_count
        except:
            return np.nan

    def calculate_physicochemical_descriptors(self, mol):
        """Calculate physicochemical descriptors"""
        descriptors = {}

        try:
            # Lipinski descriptors
            descriptors['MolecularWeight'] = Descriptors.MolWt(mol)
            descriptors['ExactMolWt'] = Descriptors.ExactMolWt(mol)
            descriptors['LogP'] = Descriptors.MolLogP(mol)
            descriptors['LogD'] = Descriptors.MolLogP(mol)  # Approximation
            descriptors['NumHDonors'] = Descriptors.NumHDonors(mol)
            descriptors['NumHAcceptors'] = Descriptors.NumHAcceptors(mol)
            descriptors['NumRotatableBonds'] = Descriptors.NumRotatableBonds(mol)

            # Advanced molecular properties
            descriptors['MolMR'] = Descriptors.MolMR(mol)  # Molar refractivity
            descriptors['LabuteASA'] = Descriptors.LabuteASA(mol)  # Accessible surface area
            descriptors['FractionCsp3'] = self._calculate_fraction_csp3(mol)  # Custom calculation
            descriptors['NumHeteroatoms'] = Descriptors.NumHeteroatoms(mol)

            # Ring descriptors with error handling
            try:
                descriptors['NumSaturatedCarbocycles'] = Descriptors.NumSaturatedCarbocycles(mol)
            except:
                descriptors['NumSaturatedCarbocycles'] = 0

            try:
                descriptors['NumSaturatedHeterocycles'] = Descriptors.NumSaturatedHeterocycles(mol)
            except:
                descriptors['NumSaturatedHeterocycles'] = 0

            try:
                descriptors['NumAliphaticCarbocycles'] = Descriptors.NumAliphaticCarbocycles(mol)
            except:
                descriptors['NumAliphaticCarbocycles'] = 0

            try:
                descriptors['NumAliphaticHeterocycles'] = Descriptors.NumAliphaticHeterocycles(mol)
            except:
                descriptors['NumAliphaticHeterocycles'] = 0

            try:
                descriptors['NumAromaticCarbocycles'] = Descriptors.NumAromaticCarbocycles(mol)
            except:
                descriptors['NumAromaticCarbocycles'] = 0

            try:
                descriptors['NumAromaticHeterocycles'] = Descriptors.NumAromaticHeterocycles(mol)
            except:
                descriptors['NumAromaticHeterocycles'] = 0

            # Ring descriptors
            descriptors['RingCount'] = Descriptors.RingCount(mol)
            descriptors['NumRings'] = Descriptors.RingCount(mol)
            descriptors['NumAromaticRings'] = Descriptors.NumAromaticRings(mol)

            # Charge descriptors
            descriptors['MaxPartialCharge'] = Descriptors.MaxPartialCharge(mol)
            descriptors['MinPartialCharge'] = Descriptors.MinPartialCharge(mol)
            descriptors['MaxAbsPartialCharge'] = Descriptors.MaxAbsPartialCharge(mol)
            descriptors['MinAbsPartialCharge'] = Descriptors.MinAbsPartialCharge(mol)

        except Exception as e:
            print(f"Error calculating physicochemical descriptors: {e}")

        return descriptors

    def calculate_3d_descriptors(self, mol):
        """Calculate 3D molecular descriptors"""
        descriptors = {}

        try:
            mol_3d = self.generate_3d_conformer(mol)
            if mol_3d is None:
                return descriptors

            # 3D descriptors with error handling
            try:
                descriptors['RadiusOfGyration'] = rdMolDescriptors.CalcRadiusOfGyration(mol_3d)
            except:
                descriptors['RadiusOfGyration'] = np.nan

            try:
                descriptors['Asphericity'] = rdMolDescriptors.CalcAsphericity(mol_3d)
            except:
                descriptors['Asphericity'] = np.nan

            try:
                descriptors['Eccentricity'] = rdMolDescriptors.CalcEccentricity(mol_3d)
            except:
                descriptors['Eccentricity'] = np.nan

            try:
                descriptors['InertialShapeFactor'] = rdMolDescriptors.CalcInertialShapeFactor(mol_3d)
            except:
                descriptors['InertialShapeFactor'] = np.nan

            try:
                descriptors['NPR1'] = rdMolDescriptors.CalcNPR1(mol_3d)
                descriptors['NPR2'] = rdMolDescriptors.CalcNPR2(mol_3d)
            except:
                descriptors['NPR1'] = np.nan
                descriptors['NPR2'] = np.nan

            try:
                descriptors['SpherocityIndex'] = rdMolDescriptors.CalcSpherocityIndex(mol_3d)
            except:
                descriptors['SpherocityIndex'] = np.nan

            # Principal moments of inertia
            try:
                pmi = rdMolDescriptors.CalcPMI1(mol_3d), rdMolDescriptors.CalcPMI2(mol_3d), rdMolDescriptors.CalcPMI3(mol_3d)
                descriptors['PMI1'] = pmi[0]
                descriptors['PMI2'] = pmi[1]
                descriptors['PMI3'] = pmi[2]
            except:
                descriptors['PMI1'] = np.nan
                descriptors['PMI2'] = np.nan
                descriptors['PMI3'] = np.nan

        except Exception as e:
            print(f"Error calculating 3D descriptors: {e}")

        return descriptors

    def calculate_fragment_descriptors(self, mol):
        """Calculate fragment-based descriptors"""
        descriptors = {}

        try:
            # Fragment counts with error handling
            fragment_functions = [
                'fr_Al_COO', 'fr_Al_OH', 'fr_Ar_COO', 'fr_Ar_OH', 'fr_COO',
                'fr_benzene', 'fr_NH0', 'fr_NH1', 'fr_NH2', 'fr_N_O',
                'fr_Ndealkylation1', 'fr_Ndealkylation2', 'fr_amide',
                'fr_aniline', 'fr_phenol'
            ]

            for frag_func in fragment_functions:
                try:
                    if hasattr(Fragments, frag_func):
                        descriptors[frag_func] = getattr(Fragments, frag_func)(mol)
                    else:
                        descriptors[frag_func] = 0
                except:
                    descriptors[frag_func] = 0

        except Exception as e:
            print(f"Error calculating fragment descriptors: {e}")

        return descriptors

    def calculate_fingerprint_descriptors(self, mol):
        """Calculate fingerprint-based descriptors"""
        descriptors = {}

        try:
            # Morgan fingerprints (ECFP)
            morgan_fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
            for i, bit in enumerate(morgan_fp):
                descriptors[f'Morgan_{i}'] = bit

            # Atom pair fingerprints
            ap_fp = Pairs.GetAtomPairFingerprintAsBitVect(mol)
            descriptors['AtomPairFP_Count'] = ap_fp.GetNumOnBits()

            # Topological torsion fingerprints
            tt_fp = Torsions.GetTopologicalTorsionFingerprintAsBitVect(mol)
            descriptors['TopologicalTorsionFP_Count'] = tt_fp.GetNumOnBits()

        except Exception as e:
            print(f"Error calculating fingerprint descriptors: {e}")

        return descriptors

    def calculate_qed_descriptors(self, mol):
        """Calculate QED and drug-likeness descriptors"""
        descriptors = {}

        try:
            # QED score
            qed_score = QED.qed(mol)
            descriptors['QED'] = qed_score

            # Individual QED components
            qed_components = QED.properties(mol)
            descriptors['QED_MW'] = qed_components.MW
            descriptors['QED_ALOGP'] = qed_components.ALOGP
            descriptors['QED_HBA'] = qed_components.HBA
            descriptors['QED_HBD'] = qed_components.HBD
            descriptors['QED_PSA'] = qed_components.PSA
            descriptors['QED_ROTB'] = qed_components.ROTB
            descriptors['QED_AROM'] = qed_components.AROM
            descriptors['QED_ALERTS'] = qed_components.ALERTS

            # Lipinski violations
            descriptors['LipinskiViolations'] = sum([
                qed_components.MW > 500,
                qed_components.ALOGP > 5,
                qed_components.HBD > 5,
                qed_components.HBA > 10
            ])

        except Exception as e:
            print(f"Error calculating QED descriptors: {e}")

        return descriptors

    def calculate_all_descriptors(self, smiles):
        """Calculate all descriptor types for a given SMILES"""
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None

        mol = self.sanitize_molecule(mol)
        if mol is None:
            return None

        all_descriptors = {}

        # Calculate different descriptor types
        descriptor_types = [
            self.calculate_topological_descriptors,
            self.calculate_physicochemical_descriptors,
            self.calculate_3d_descriptors,
            self.calculate_fragment_descriptors,
            self.calculate_qed_descriptors
        ]

        for descriptor_func in descriptor_types:
            descriptors = descriptor_func(mol)
            all_descriptors.update(descriptors)

        # Add fingerprint descriptors (optional - creates many features)
        # fingerprint_descriptors = self.calculate_fingerprint_descriptors(mol)
        # all_descriptors.update(fingerprint_descriptors)

        return all_descriptors

    def generate_descriptors_from_file(self, file_path, smiles_column='SMILES', activity_column='pIC50'):
        """Generate descriptors from a CSV file"""
        print("Loading data...")
        data = pd.read_csv(file_path)

        descriptor_data = []
        failed_count = 0

        print("Generating descriptors...")
        for index, row in tqdm(data.iterrows(), total=len(data)):
            smiles = row[smiles_column]
            activity = row[activity_column] if activity_column in row else None

            descriptors = self.calculate_all_descriptors(smiles)

            if descriptors is not None:
                descriptors['SMILES'] = smiles
                if activity is not None:
                    descriptors['pIC50'] = activity
                descriptor_data.append(descriptors)
            else:
                failed_count += 1
                self.failed_molecules.append(smiles)

        print(f"Successfully processed {len(descriptor_data)} molecules")
        print(f"Failed to process {failed_count} molecules")

        # Convert to DataFrame
        descriptor_df = pd.DataFrame(descriptor_data)

        # Handle missing values
        descriptor_df = descriptor_df.fillna(0)

        return descriptor_df

    def save_descriptors(self, descriptor_df, output_path):
        """Save descriptors to CSV file"""
        descriptor_df.to_csv(output_path, index=False)
        print(f"Descriptors saved to {output_path}")

        # Save feature names
        feature_names = [col for col in descriptor_df.columns if col not in ['SMILES', 'pIC50']]
        with open(output_path.replace('.csv', '_features.txt'), 'w') as f:
            for feature in feature_names:
                f.write(f"{feature}\n")

        print(f"Feature names saved to {output_path.replace('.csv', '_features.txt')}")

        # Print statistics
        print(f"\nDescriptor Statistics:")
        print(f"Total compounds: {len(descriptor_df)}")
        print(f"Total descriptors: {len(feature_names)}")
        print(f"Failed molecules: {len(self.failed_molecules)}")

# File upload and download functionality
from google.colab import files
import os

def upload_and_process_file():
    """Upload file and process it"""
    print("Please upload your CSV file containing SMILES and pIC50 columns:")
    uploaded = files.upload()

    # Get the uploaded file name
    file_name = list(uploaded.keys())[0]
    print(f"Uploaded file: {file_name}")

    # Initialize descriptor generator
    desc_gen = AdvancedDescriptorGenerator()

    # Generate descriptors
    try:
        descriptor_df = desc_gen.generate_descriptors_from_file(
            file_name,
            smiles_column='SMILES',
            activity_column='pIC50'
        )

        # Save descriptors
        output_path = 'Final_descriptors_for_QSAR.csv'
        desc_gen.save_descriptors(descriptor_df, output_path)

        # Display basic statistics
        print("\nDataset Overview:")
        print(descriptor_df.head())
        print(f"\nDataset shape: {descriptor_df.shape}")

        # Check for missing values
        missing_values = descriptor_df.isnull().sum()
        print(f"\nMissing values: {missing_values.sum()}")

        # Download the results
        print("\nDownloading results...")
        files.download(output_path)
        files.download(output_path.replace('.csv', '_features.txt'))

        return descriptor_df

    except Exception as e:
        print(f"Error: {e}")
        print("Make sure your CSV file has columns named 'SMILES' and 'pIC50'")

# Main execution
if __name__ == "__main__":
    # Run the upload and process function
    upload_and_process_file()

Please upload your CSV file containing SMILES and pIC50 columns:


Saving data_5.csv to data_5.csv
Uploaded file: data_5.csv
Loading data...
Generating descriptors...


100%|██████████| 804/804 [06:50<00:00,  1.96it/s]

Successfully processed 804 molecules
Failed to process 0 molecules
Descriptors saved to Final_descriptors_for_QSAR.csv
Feature names saved to Final_descriptors_for_QSAR_features.txt

Descriptor Statistics:
Total compounds: 804
Total descriptors: 76
Failed molecules: 0

Dataset Overview:
   WienerIndex  ZagrebIndex1  ZagrebIndex2  BalabanJ    TPSA     Kappa1  \
0      20530.0           306           343  1.796149  373.45  51.798145   
1      15110.0           274           307  2.244185  318.75  47.082570   
2      19754.0           300           331  2.434699  396.65  53.066906   
3      16615.0           272           297  3.280906  389.31  51.018868   
4      11438.0           250           280  2.262341  309.88  42.586926   

      Kappa2     Kappa3       Chi0       Chi1  ...  QED_ALOGP  QED_HBA  \
0  25.532052  17.990998  46.691295  29.807990  ...    -2.2658       15   
1  21.291219  15.583601  42.490470  25.980868  ...    -1.3917       12   
2  26.414621  20.568806  47.277081  29.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>